# Same test set comparison


In [ ]:
%pip install -q \
  torch==2.13.0 \
  transformers==5.16.1 \
  datasets==5.0.1 \
  accelerate==1.14.0 \
  peft==0.20.0 \
  trl==1.12.0 \
  bitsandbytes==0.50.2 \
  evaluate==0.4.6 \
  requests tqdm sentencepiece huggingface_hub pandas scikit-learn

%pip install -q sentence-transformers

In [ ]:
import json, random, re, string, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from datasets import Dataset, DatasetDict

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_REPO="https://github.com/Gokcimen/Home_Appliance_Dataset"
!rm -rf /content/Home_Appliance_Dataset
!git clone -q --depth 1 {DATA_REPO}.git /content/Home_Appliance_Dataset

DATA_ROOT=Path("/content/Home_Appliance_Dataset")

def flatten(path):
    raw=json.loads(Path(path).read_text(encoding="utf-8"))
    rows=[]
    for article in raw["data"]:
        title=article["title"]
        for para in article["paragraphs"]:
            context=para["context"]
            for qa in para["qas"]:
                rows.append({
                    "id":str(qa["id"]),
                    "title":title,
                    "context":context,
                    "question":qa["question"],
                    "answers":{
                        "text":[a["text"] for a in qa["answers"]],
                        "answer_start":[int(a["answer_start"]) for a in qa["answers"]],
                    }
                })
    return rows

raw_datasets=DatasetDict({
    "train":Dataset.from_list(flatten(DATA_ROOT/"train.json")),
    "validation":Dataset.from_list(flatten(DATA_ROOT/"dev.json")),
    "test":Dataset.from_list(flatten(DATA_ROOT/"test.json")),
})

assert len(raw_datasets["train"])==8000
assert len(raw_datasets["validation"])==1000
assert len(raw_datasets["test"])==1000

ids={s:set(raw_datasets[s]["id"]) for s in raw_datasets}
assert not ids["train"]&ids["validation"]
assert not ids["train"]&ids["test"]
assert not ids["validation"]&ids["test"]
all_ids=set().union(*ids.values())
assert len(all_ids)==10000
assert {int(x) for x in all_ids}==set(range(1,10001))

titles={s:set(raw_datasets[s]["title"]) for s in raw_datasets}
assert not titles["train"]&titles["validation"]
assert not titles["train"]&titles["test"]
assert not titles["validation"]&titles["test"]
assert len(set().union(*titles.values()))==1111

print("train",len(raw_datasets["train"]))
print("validation",len(raw_datasets["validation"]))
print("test",len(raw_datasets["test"]))
print("products",len(set().union(*titles.values())))


In [ ]:
def normalize_answer(text):
    text=str(text).lower()
    text="".join(c for c in text if c not in string.punctuation)
    text=re.sub(r"\b(a|an|the)\b"," ",text)
    return " ".join(text.split())

def exact_match(pred,gold):
    return float(normalize_answer(pred)==normalize_answer(gold))

def token_f1(pred,gold):
    p=normalize_answer(pred).split()
    g=normalize_answer(gold).split()
    if not p or not g:
        return float(p==g)
    same=sum((Counter(p)&Counter(g)).values())
    if same==0:
        return 0.0
    precision=same/len(p)
    recall=same/len(g)
    return 2*precision*recall/(precision+recall)

def score_rows(rows):
    return {
        "n":len(rows),
        "EM":100*sum(exact_match(r["prediction"],r["gold"]) for r in rows)/len(rows),
        "F1":100*sum(token_f1(r["prediction"],r["gold"]) for r in rows)/len(rows),
        "mean_latency_seconds":sum(float(r.get("latency_seconds",0)) for r in rows)/len(rows),
    }


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

train_rows=[raw_datasets["train"][i] for i in range(len(raw_datasets["train"]))]

kg_questions=[r["question"] for r in train_rows]
kg_answers=[r["answers"]["text"][0] for r in train_rows]
kg_embeddings=embedder.encode(kg_questions,normalize_embeddings=True,show_progress_bar=True)

train_context_by_title={}
for r in train_rows:
    train_context_by_title[r["title"]]=r["context"]

rag_titles=list(train_context_by_title)
rag_docs=[train_context_by_title[t] for t in rag_titles]
rag_embeddings=embedder.encode(rag_docs,normalize_embeddings=True,show_progress_bar=True)

def kg_only_answer(question):
    q=embedder.encode([question],normalize_embeddings=True)[0]
    idx=int(np.argmax(kg_embeddings@q))
    return kg_answers[idx]

def rag_retrieve(question,top_k=4):
    q=embedder.encode([question],normalize_embeddings=True)[0]
    idx=np.argsort(rag_embeddings@q)[::-1][:top_k]
    return [rag_docs[i] for i in idx]

print("KG train questions",len(kg_questions))
print("RAG train documents",len(rag_docs))


In [ ]:
import json
from pathlib import Path

EXPECTED_TEST_IDS=set(raw_datasets["test"]["id"])

def load_prediction_file(path):
    rows=[json.loads(x) for x in Path(path).read_text().splitlines() if x.strip()]
    assert len(rows)==1000
    ids={str(r["id"]) for r in rows}
    assert ids==EXPECTED_TEST_IDS
    return rows

def compare(method_to_file):
    output=[]
    for method,path in method_to_file.items():
        rows=load_prediction_file(path)
        metrics=score_rows(rows)
        metrics["method"]=method
        output.append(metrics)
    return pd.DataFrame(output)
